<a href="https://colab.research.google.com/github/Adigozalovh/DeepLearning/blob/main/Flood_Area_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader

In [34]:
class Conv3(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()
    self.calc = nn.Sequential(
        nn.Conv2d(n_features, out_channel, 3, 1, 1),
        nn.ReLU(),
        nn.Conv2d(out_channel, out_channel, 3, 1, 1),
        nn.ReLU()
    )
  def forward(self, X):
    return self.calc(X)

In [35]:
class UpConv(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()
    self.upconv = nn.ConvTranspose2d(n_features, out_channel, kernel_size=2, stride = 2)
  def forward(self, X):
    return self.upconv(X)

In [36]:
class UNet(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()

    self.max_pool = nn.MaxPool2d(2, 2)

    self.DownConv1 = Conv3(n_features, 64)
    self.DownConv2 = Conv3(64, 128)
    self.DownConv3 = Conv3(128, 256)
    self.DownConv4 = Conv3(256, 512)

    self.Bottleneck = Conv3(512, 1024)

    self.UpConv1 = UpConv(1024, 512)
    self.UpConv2 = UpConv(512, 256)
    self.UpConv3 = UpConv(256, 128)
    self.UpConv4 = UpConv(128, 64)

    self.Expansion1 = Conv3(1024, 512)
    self.Expansion2 = Conv3(512, 256)
    self.Expansion3 = Conv3(256, 128)
    self.Expansion4 = Conv3(128, 64)

    self.output = nn.Conv2d(64, out_channel, 1, stride = 1)

  def forward(self, X):
    X = self.DownConv1(X)
    skip1 = X
    X = self.max_pool(X)
    X = self.DownConv2(X)
    skip2 = X
    X = self.max_pool(X)
    X = self.DownConv3(X)
    skip3 =X
    X = self.max_pool(X)
    X = self.DownConv4(X)
    skip4 = X
    X = self.max_pool(X)
    X = self.Bottleneck(X)

    X = self.UpConv1(X)
    X = torch.cat([X, skip4], dim = 1)
    X = self.Expansion1(X)
    X = self.UpConv2(X)
    X = torch.cat([X, skip3], dim = 1)
    X = self.Expansion2(X)
    X = self.UpConv3(X)
    X = torch.cat([X, skip2], dim = 1)
    X = self.Expansion3(X)
    X = self.UpConv4(X)
    X = torch.cat([X, skip1], dim = 1)
    X = self.Expansion4(X)

    X = self.output(X)

    return X

In [37]:
def dice_score(pred, label, eps=1e-8):
  proba = torch.sigmoid(pred)
  preds = (proba > 0.5).float()

  preds = preds.view(-1)
  labels = label.view(-1)

  intersection = (preds * labels).sum()
  dice_score = (2.0 * intersection + eps) / (preds.sum() + labels.sum() + eps)

  return dice_score

In [32]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [43]:
def train_model(model, train_loader, val_loader, criterion, optimizer, n_epochs, device):
    model.to(device)

    for epoch in range(n_epochs):
        model.train()
        total_train_loss = 0
        total_train_dice = 0

        for images, masks in train_loader:
            images = images.to(device).float().permute(0, 3, 1, 2) / 255.0
            if masks.max() > 1:
                masks = masks.to(device).float().unsqueeze(1) / 255.0
            else:
                masks = masks.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            total_train_dice += dice_score(outputs, masks).item()

        model.eval()
        total_val_loss = 0
        total_val_dice = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device).float().permute(0, 3, 1, 2) / 255.0
                if masks.max() > 1:
                    masks = masks.to(device).float().unsqueeze(1) / 255.0
                else:
                    masks = masks.to(device).float().unsqueeze(1)

                outputs = model(images)
                loss = criterion(outputs, masks)
                total_val_loss += loss.item()
                total_val_dice += dice_score(outputs, masks).item()
    print(f"Epoch [{epoch+1}/{n_epochs}]")
    print(f"Train | Loss: {total_train_loss/len(train_loader):.4f}, Dice: {total_train_dice/len(train_loader):.4f}")
    print(f"Validation | Loss: {total_val_loss/len(val_loader):.4f}, Dice: {total_val_dice/len(val_loader):.4f}")

In [21]:
!curl -L -o flood-area-segmentation.zip https://www.kaggle.com/api/v1/datasets/download/faizalkarim/flood-area-segmentation

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  106M  100  106M    0     0   119M      0 --:--:-- --:--:-- --:--:--  119M


In [22]:
!unzip /content/flood-area-segmentation.zip

Archive:  /content/flood-area-segmentation.zip
replace Image/0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: Image/0.jpg             
  inflating: Image/1.jpg             
  inflating: Image/10.jpg            
  inflating: Image/1000.jpg          
  inflating: Image/1001.jpg          
  inflating: Image/1002.jpg          
  inflating: Image/1003.jpg          
  inflating: Image/1004.jpg          
  inflating: Image/1005.jpg          
  inflating: Image/1006.jpg          
  inflating: Image/1007.jpg          
  inflating: Image/1008.jpg          
  inflating: Image/1009.jpg          
  inflating: Image/1010.jpg          
  inflating: Image/1011.jpg          
  inflating: Image/1012.jpg          
  inflating: Image/1013.jpg          
  inflating: Image/1014.jpg          
  inflating: Image/1015.jpg          
  inflating: Image/1016.jpg          
  inflating: Image/1017.jpg          
  inflating: Image/1018.jpg          
  inflating: Image/1019.jpg          
  inflating: Image

In [39]:
import os
import glob
from PIL import Image

In [40]:
class SegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask

In [41]:
import numpy as np

def load_data(img_dir, mask_dir):
    images = []
    masks = []
    img_names = sorted(os.listdir(img_dir))
    mask_names = sorted(os.listdir(mask_dir))

    for img_name, mask_name in zip(img_names, mask_names):
        img = Image.open(os.path.join(img_dir, img_name)).convert("RGB").resize((128, 128))
        mask = Image.open(os.path.join(mask_dir, mask_name)).convert("L").resize((128, 128))
        images.append(np.array(img))
        masks.append(np.array(mask))

    return np.array(images), np.array(masks)

In [44]:
img_path = "/content/Image"
mask_path = "/content/Mask"
X, y = load_data(img_path, mask_path)

transform = transforms.Compose([transforms.ToTensor()])
dataset = SegmentationDataset(X, y, transform=None)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.dataset.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)


model = UNet(n_features=3, out_channel=1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

train_model(model, train_loader, val_loader, criterion, optimizer, n_epochs=10, device=device)

Epoch [10/10]
Train | Loss: 0.4493, Dice: 0.7328
Validation | Loss: 0.4278, Dice: 0.7587
